# Simulating the non-deterministic controller (issue #51)

This notebook shows how to

1. build a universal co-Büchi automaton (UCB) from an LTL formula,
2. solve the resulting safety game with acacia-bonsai,
3. enumerate every winning and losing output valuation at a given state,
4. step the simulation forward under a user-chosen IO,
5. and, when a chosen output falls outside the current winning region, try to grow $k$ and see if a larger winning region now covers the state.

The spec is the classic GR(1) request/grant: `G F req -> G F grant`. We build the UCB for the **negation** (which is what the CLI does for realisability checks), so the safety game is “can the controller keep this bad-trace formula falsified?”.

In [ ]:
import itertools

import acacia_boomslang as ab
import spot

SPEC = "!((G (F (req))) -> (G (F (grant))))"
INPUTS  = ["req"]
OUTPUTS = ["grant"]

INITIAL_K = 2
K_STEP    = 3
MAX_K     = 30

## Build the UCB and inspect it

`create_twa` returns a `Game` that bundles the UCB, the BDD dictionary and the input/output APs. We can render the underlying automaton by handing the HOA serialisation back to spot.

In [ ]:
game = ab.create_twa(SPEC, INPUTS, OUTPUTS)
print(f"UCB: {ab.num_states(game)} states, initial = {ab.initial_state_number(game)}")

aut = spot.automaton(ab.get_aut_hoa(game))
aut

## Solve the safety game

The two preprocessing calls and `set_bool_thresh_no_bool_states` mirror the CLI’s default pipeline.

In [ ]:
def solve(game, k):
    ab.preprocess_aut_standard(game, k_max=k)
    ab.set_bool_thresh_no_bool_states(game, k_max=k)
    return ab.solve_acacia_safety_game(game, k_max=k, k_min=min(2, k), k_inc=K_STEP)

k = INITIAL_K
result = solve(game, k)
assert result.is_real(), "unrealisable at INITIAL_K— increase it"

winreg = result.get_winning_region()
print(f"Realizable at k={k}; winning region has {len(winreg)} antichain heads.")
for i, v in enumerate(winreg):
    print(f"  head {i}:", list(v))

## Classify output choices at a state

Given a state vector `v` and a fixed input valuation, we enumerate every output assignment and split them by whether the one-step successor is still in the winning region.

The set of *winning* outputs is exactly the non-deterministic controller at this state.

In [ ]:
def all_assignments(aps):
    for bits in itertools.product([False, True], repeat=len(aps)):
        t = [a for a, b in zip(aps, bits) if b]
        f = [a for a, b in zip(aps, bits) if not b]
        yield t, f

def label(t, f):
    return " & ".join([*t, *(f"!{a}" for a in f)]) or "tt"

def classify_outputs(game, v, true_inputs, false_inputs, winreg, k_cap):
    winning, losing = [], []
    for t_out, f_out in all_assignments(OUTPUTS):
        s = ab.successor(
            game, v,
            ab.StringVector(true_inputs + t_out),
            ab.StringVector(false_inputs + f_out),
            k_cap)
        (winning if winreg.contains(s) else losing).append((t_out, f_out, s))
    return winning, losing

v = ab.get_initial_state(game)
print("state vector:", [v[i] for i in range(len(v))])

winning, losing = classify_outputs(game, v, ["req"], [], winreg, k)
print("under input req=T, grant should be one of the winning outputs:")
for t, f, s in winning:
    print(f"  W  {label(t, f):>12}  -> {[s[i] for i in range(len(s))]}")
for t, f, s in losing:
    print(f"  L  {label(t, f):>12}  -> {[s[i] for i in range(len(s))]}")

## One simulation step

Pick any output (winning or losing) and compute the next state vector. Below we pick the first winning output under `req=T` and update `v`.

In [ ]:
t_out, f_out, succ = winning[0]
print(f"pick output {label(t_out, f_out)} -> successor {[succ[i] for i in range(len(succ))]}")
v = succ

## Growing $k$ when a losing output is picked

If the user picks a losing output, the successor is outside the current winning region. We re-solve with a larger $k$ and see if the new (larger) winning region covers the state.

For the GR(1) request/grant spec the winning region is already saturated at `k=2` (the controller just needs one bit of memory), so the retry loop only demonstrates the API mechanics. Try with richer specs (e.g. bounded response `!G(req -> (grant | X grant | XX grant))`) to see $k$ actually matter.

In [ ]:
def try_grow_k(game, escaped_vec, current_k):
    new_k = current_k
    while new_k + K_STEP <= MAX_K:
        new_k += K_STEP
        fresh_game = ab.create_twa(SPEC, INPUTS, OUTPUTS)
        fresh_result = solve(fresh_game, new_k)
        if not fresh_result.is_real():
            continue
        fresh_w = fresh_result.get_winning_region()
        recast = ab.make_vector(fresh_game, ab.IntVector([escaped_vec[i] for i in range(len(escaped_vec))]))
        if fresh_w.contains(recast):
            return fresh_game, fresh_w, recast, new_k
    return None

# Force a "losing" step by picking a (hypothetically) bad output if any losing
# choices exist at this state.
winning, losing = classify_outputs(game, v, [], ["req"], winreg, k)
print("under input !req, winning outputs:")
for t, f, s in winning:
    print("  W", label(t, f), "->", [s[i] for i in range(len(s))])
print("losing outputs:")
for t, f, s in losing:
    print("  L", label(t, f), "->", [s[i] for i in range(len(s))])

if losing:
    t, f, s = losing[0]
    print(f"\npicking losing output {label(t, f)} -> {[s[i] for i in range(len(s))]}")
    grown = try_grow_k(game, s, k)
    if grown:
        game, winreg, v, k = grown
        print(f"recovered at k={k}; winning region has {len(winreg)} heads.")
    else:
        print("no larger k up to MAX_K recovers this state.")
else:
    print("no losing outputs at this state under the chosen input.")

## Putting it together

The `python_examples/example_simulate.py` script packages this into an interactive loop with prompts for inputs and outputs. Pass `auto` to drive it non-interactively as a smoke test:

```shell
python3 python_examples/example_simulate.py auto
```